In [1]:
import os
os.environ["RAY_DEDUP_LOGS"] = "0"
os.environ["RAY_DISABLE_METRICS"] = "1"
os.environ["RAY_DISABLE_METRICS_EXPORT"] = "1"

In [2]:

import polars as pl
from sklearn.metrics import accuracy_score

from autotsc.utils import load_dataset
from time import perf_counter

import numpy as np
import polars as pl
import ray
from aeon.classification.dictionary_based import REDCOMETS
from aeon.classification.interval_based import SupervisedTimeSeriesForest

from aeon.classification.base import BaseClassifier
from aeon.classification.convolution_based import (
    MiniRocketClassifier,
    MultiRocketClassifier,
    RocketClassifier,
)
from aeon.classification.dummy import DummyClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
    SummaryClassifier,
)
from aeon.classification.interval_based import (
    QUANTClassifier,
)
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegressionCV
from aeon.classification.sklearn import SklearnClassifierWrapper
from aeon.transformations.collection import DownsampleTransformer
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
import types
from autotsc.models import RidgeClassifierCVWithProba
from aeon.classification.convolution_based import HydraClassifier
from autotsc import transformers, utils
from aeon.classification.dictionary_based import IndividualBOSS
from aeon.classification.interval_based import (
    RSTSF,
    CanonicalIntervalForestClassifier,
    DrCIFClassifier,
    QUANTClassifier,
    RandomIntervalSpectralEnsembleClassifier,
    SupervisedTimeSeriesForest,
    TimeSeriesForestClassifier,
)
from aeon.classification.interval_based import RSTSF
from aeon.classification.dictionary_based import WEASEL_V2
from aeon.classification.dictionary_based import ContractableBOSS
from aeon.classification.hybrid import RISTClassifier
from aeon.pipeline import make_pipeline as aeon_make_pipeline
import hvplot.polars
import holoviews as hv
from aeon.classification.convolution_based import MultiRocketHydraClassifier
from autotsc.utils import *
from autotsc.models import *

2025-11-27 12:14:21.727743: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
X_train, y_train, X_test, y_test = load_dataset("CricketZ")
#X_train, y_train, X_test, y_test = load_dataset("MelbournePedestrian")
#X_train, y_train, X_test, y_test = load_dataset("Worms")
# X_train, y_train, X_test, y_test = load_dataset("PhalangesOutlinesCorrect")
# X_train, y_train, X_test, y_test = load_dataset("Lightning2")
# X_train, y_train, X_test, y_test = load_dataset("CricketY")
#X_train, y_train, X_test, y_test = load_dataset("OSULeaf")
#X_train, y_train, X_test, y_test = load_dataset("Wine")
#X_train, y_train, X_test, y_test = load_dataset("FiftyWords")
#X_train, y_train, X_test, y_test = load_dataset("Haptics")

	
#X_train, y_train, X_test, y_test = load_dataset("InlineSkate")
X_train, y_train, X_test, y_test = load_dataset("SemgHandSubjectCh2")

In [4]:

#

In [6]:
with utils.ray_init_or_reuse(num_cpus=24, resources={"meta": 100}, ignore_reinit_error=True):
    model = AutoTSCModel()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    accuracy_score(y_test, pred)

2025-11-27 12:14:27,466	INFO worker.py:2023 -- Started a local Ray instance.
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


|------------------------|
| Number of samples: 450 |
| Number of channels: 1  |
| Length of series: 1500 |
| Number of classes: 5   |
| CPUs: 24/24            |
| GPUs: 2/2              |
| Random seed: 8585      |
| Number of folds: 20    |
|------------------------|


(ray_run_fit_predict_proba_wrapper pid=2778443) 2025-11-27 12:14:32.036915: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(ray_run_fit_predict_proba_wrapper pid=2778443) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(ray_run_fit_predict_proba_wrapper pid=2778450) 2025-11-27 12:14:32.036916: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(ray_run_fit_predict_proba_wrapper pid=2778450) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(ray_run_model_on_fold pid=2778464) 2025-11-27 12:14:39.140681: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructio

(ray_run_fit_predict_proba_wrapper pid=2778443) Completed fit_predict_proba in Ray task in 8.56s
(ray_run_fit_predict_proba_wrapper pid=2778450) Completed fit_predict_proba in Ray task in 10.31s
Trained base model tab-ridge, OOF accuracy: 0.6733 in 0.04s
Trained base model tab-rf, OOF accuracy: 0.7467 in 2.01s
Trained meta model m-ridgecv, OOF accuracy: 0.8200 in 0.04s
Trained meta model m-svm, OOF accuracy: 0.8178 in 0.19s


(pid=gcs_server) [2025-11-27 12:14:56,088 E 2778175 2778175] (gcs_server) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2025-11-27 12:14:57,393 E 2778358 2778358] (raylet) main.cc:979: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(ray_run_fit_predict_proba_wrapper pid=2778443) [2025-11-27 12:14:59,378 E 2778443 2778616] core_worker_process.cc:837: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(ray_run_fit_predict_proba_wrapper pid=2778450) [2025-11-27 12:14:59,367 E 2778450 2778595] core_worker_process.c

Trained meta model m-log, OOF accuracy: 0.8244 in 18.01s


In [7]:
summary = model.summary()
summary

model_id,model,validation_accuracy,stacking_level,train_time
str,str,f64,i64,f64
"""tab-ridge""","""RayCrossValidationWrapper(mode…",0.673333,0,0.035432
"""tab-rf""","""RayCrossValidationWrapper(mode…",0.746667,0,2.014401
"""m-ridgecv""","""Ensemble(RidgeClassifierCVWith…",0.82,1,null
"""m-svm""","""Ensemble(SVC(kernel='linear',p…",0.817778,1,null
"""m-log""","""Ensemble(LogisticRegressionCV(…",0.824444,1,null


In [8]:
test_accs = []
preds = model.predict_per_model(X_test)
for m, p in preds.items():
    acc = accuracy_score(y_test, p)
    test_accs.append({
        "model_id": m,
        "test_accuracy": acc,
    })
test_accs = pl.DataFrame(test_accs)
summary = summary.join(test_accs, on="model_id")

2025-11-27 12:15:05,569	INFO worker.py:2023 -- Started a local Ray instance.
(ray_run_predict_proba pid=2782881) 2025-11-27 12:15:10.993627: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(ray_run_predict_proba pid=2782881) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(ray_run_predict_proba pid=2782864) 2025-11-27 12:15:10.987115: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(ray_run_predict_proba pid=2782864) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(ray_run_predict_proba pid=2782863) 2025-11-27 12:15:11.010102: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to

In [9]:
pl.Config.set_fmt_str_lengths(500)
for r in summary.sort("validation_accuracy").iter_rows():
    print(r)

('tab-ridge', 'RayCrossValidationWrapper(model=SklearnClassifierWrapper(classifier=RidgeClassifierCVWithProba(alphas=array([1.00000000e-03,4.64158883e-03,2.15443469e-02,1.00000000e-01,4.64158883e-01,2.15443469e+00,1.00000000e+01,4.64158883e+01,2.15443469e+02,1.00000000e+03]))))', 0.6733333333333333, 0, 0.035432401948492044, 0.6355555555555555)
('tab-rf', 'RayCrossValidationWrapper(model=SklearnClassifierWrapper(classifier=RandomForestClassifier(n_estimators=500,n_jobs=-1)))', 0.7466666666666667, 0, 2.0144009637006093, 0.7088888888888889)
('m-svm', "Ensemble(SVC(kernel='linear',probability=True))", 0.8177777777777778, 1, None, 0.7088888888888889)
('m-ridgecv', 'Ensemble(RidgeClassifierCVWithProba(alphas=array([1.00000000e-03,4.64158883e-03,2.15443469e-02,1.00000000e-01,4.64158883e-01,2.15443469e+00,1.00000000e+01,4.64158883e+01,2.15443469e+02,1.00000000e+03])))', 0.82, 1, None, 0.6955555555555556)
('m-log', 'Ensemble(LogisticRegressionCV(cv=5,n_jobs=-1))', 0.8244444444444444, 1, None, 0

In [10]:
# Create the scatter plot
scatter = summary.hvplot.scatter(
    x="validation_accuracy",
    y="test_accuracy",
    c="stacking_level",
    hover_cols=["model_id", "validation_accuracy", "test_accuracy", "stacking_level"],
    cmap="viridis",
    size=80,
    alpha=0.7,
    title="Validation vs Test Accuracy per Model",
    xlabel="Validation Accuracy",
    ylabel="Test Accuracy",
    width=700,
    height=600,
    grid=True  # Add grid here
)

# Add text labels for model_id
labels = hv.Labels(
    summary.select(["validation_accuracy", "test_accuracy", "model_id"]).to_pandas(),
    kdims=["validation_accuracy", "test_accuracy"],
    vdims=["model_id"]
).opts(
    text_font_size="8pt",
    text_align="left",
    text_baseline="bottom",
    xoffset=0.0005,  # slight offset so text doesn't overlap point
    yoffset=0.0005
)

# Add diagonal line
min_val = min(summary["validation_accuracy"].min(), summary["test_accuracy"].min())
max_val = max(summary["validation_accuracy"].max(), summary["test_accuracy"].max())
diagonal = hv.Curve([(min_val, min_val), (max_val, max_val)]).opts(
    color="gray", 
    line_dash="dashed",
    line_width=1
)

# Combine all
plot = scatter * labels * diagonal
plot

:Overlay
   .Scatter.I :Scatter   [validation_accuracy]   (test_accuracy,stacking_level,model_id)
   .Labels.I  :Labels   [validation_accuracy,test_accuracy]   (model_id)
   .Curve.I   :Curve   [x]   (y)

(pid=gcs_server) [2025-11-27 12:15:34,154 E 2782628 2782628] (gcs_server) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2025-11-27 12:15:35,491 E 2782774 2782774] (raylet) main.cc:979: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(ray_run_predict_proba pid=2782870) [2025-11-27 12:15:37,591 E 2782870 2783356] core_worker_process.cc:837: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(ray_run_predict_proba pid=2782858) [2025-11-27 12:15:37,584 E 2782858 2783316] core_worker_process.cc:837: Failed to establi